In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
from io import BytesIO
import time
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.action_chains import ActionChains
import pyautogui
import random
from tqdm import tqdm


In [7]:
import os
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

url = "https://www.1001fonts.com/handwritten-fonts.html?page=1"

# Setup Chrome options for Windows
options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--headless')  # Run in headless mode for automation

# Use webdriver-manager to automatically download and manage chromedriver
webdriver_service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=webdriver_service, options=options)

print("Chrome driver initialized successfully")


Chrome driver initialized successfully


In [8]:
base_url = "https://www.flaconi.de/locken-pflege/?offset="

base_url = "https://www.1001fonts.com/handwritten-fonts.html?page="
pages = range(1, 1218, 1)

page_links = [base_url + str(page) for page in pages]


In [9]:
from bs4 import BeautifulSoup
from tqdm import tqdm
import csv
import os

data = []

# Create output directory
output_dir = r"E:\Projects\HTR_PR_Lab\HTR-Pipeline\data\synthetic"
os.makedirs(output_dir, exist_ok=True)

# Iterate over each URL in the page_links list
for url in tqdm(page_links):
    # Open the webpage with Selenium
    driver.get(url)
    time.sleep(1)  # Give page time to load
    
    # Parse the page content using BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # Find the font list
    font_list = soup.find('ul', {'class': 'font-list', 'id': 'typeface-browsing-list'})
    if font_list:
        # Find all the font items
        font_items = font_list.find_all('li', {'class': 'font-list-item font-preview'})
        for item in font_items:
            # Extract the download link - use lambda to match class containing 'btn-success'
            download_link_tag = item.find('a', {'role': 'button'}, class_=lambda x: x and 'btn-success' in x if x else False)
            if download_link_tag and download_link_tag.has_attr('href'):
                link = download_link_tag.get('href')
                # Find the use type
                license_element = item.find('a', class_=lambda x: x and 'license-yes' in x if x else False)
                if license_element:
                    use_type = 'commercial use'
                else:
                    license_element = item.find('a', class_=lambda x: x and 'license-no' in x if x else False)
                    if license_element:
                        use_type = 'personal use'
                    else:
                        use_type = 'unknown'
                data.append({'link': link, 'use_type': use_type})
            else:
                # If the download link is not found, skip this item
                continue

# Close the driver
driver.quit()

# Save the data to a CSV file in the synthetic data directory
output_file = os.path.join(output_dir, 'font_links_license.csv')
with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['link', 'use_type']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for entry in data:
        writer.writerow(entry)

# Print summary
print(f"\nTotal fonts found: {len(data)}")
print(f"Output saved to: {output_file}")
print(f"\nFirst 5 entries:")
for entry in data[:5]:
    print(entry)


100%|██████████| 1217/1217 [35:37<00:00,  1.76s/it]



Total fonts found: 14604
Output saved to: E:\Projects\HTR_PR_Lab\HTR-Pipeline\data\synthetic\font_links_license.csv

First 5 entries:
{'link': '/download/rustic-roadway-personal-use.zip', 'use_type': 'personal use'}
{'link': '/download/priestacy.zip', 'use_type': 'personal use'}
{'link': '/download/rockybilly.zip', 'use_type': 'personal use'}
{'link': '/download/moralana.zip', 'use_type': 'personal use'}
{'link': '/download/lucy-said-ok-personal-use.zip', 'use_type': 'personal use'}


In [ ]:
now I would like to find this one.
There are two types  "commercial use" and "personal use"

<a href="/scriptina-font.html#license" class="btn btn-link license-yes me-2" data-bs-toggle="tooltip" data-bs-html="true" aria-disabled="true" aria-label="This font is free for<br>commercial use" data-bs-original-title="This font is free for<br>commercial use"><i class="fa fo-money"></i> </a>

<a href="/billy-argel-font-font.html#license" class="btn btn-link license-no me-2" data-bs-toggle="tooltip" data-bs-html="true" aria-disabled="true" aria-label="This font is free for<br>personal use" data-bs-original-title="This font is free for<br>personal use"><i class="fa fo-money"></i> </a>

So I'd like to map each link to this type "commercial use" or "personal use"

All fonts are under here : 
<ul class="font-list" id="typeface-browsing-list">

and all the font items are under here : 
<li class="font-list-item font-preview">

In [5]:
# Debug: Let's check what we're actually getting from a single page
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# Setup
options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
# Remove headless to see what's happening
# options.add_argument('--headless')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Test with first page
test_url = "https://www.1001fonts.com/handwritten-fonts.html?page=1"
driver.get(test_url)
time.sleep(3)  # Wait for page to load

soup = BeautifulSoup(driver.page_source, 'html.parser')

# Check if font list exists
font_list = soup.find('ul', {'class': 'font-list', 'id': 'typeface-browsing-list'})
print(f"Font list found: {font_list is not None}")

if font_list:
    font_items = font_list.find_all('li', {'class': 'font-list-item font-preview'})
    print(f"Number of font items: {len(font_items)}")
    
    if len(font_items) > 0:
        # Check first item structure
        first_item = font_items[0]
        print("\nFirst item HTML structure:")
        print(first_item.prettify()[:1000])
else:
    # Look for alternative structures
    print("\nSearching for alternative structures...")
    all_lis = soup.find_all('li')
    print(f"Total <li> elements: {len(all_lis)}")
    
    # Look for download buttons
    download_buttons = soup.find_all('a', {'role': 'button'})
    print(f"Download buttons found: {len(download_buttons)}")
    
    if len(download_buttons) > 0:
        print("\nFirst download button:")
        print(download_buttons[0].prettify())

driver.quit()


Font list found: True
Number of font items: 12

First item HTML structure:
<li aria-labelledby="rustic-roadway-personal-use-list-item" class="font-list-item font-preview">
 <header>
  <span class="font-title muted-label" id="rustic-roadway-personal-use-list-item">
   Rustic Roadway - Personal use
   <small class="text-muted">
    <span>
     by
    </span>
    <picture class="user-avatar-container">
     <source srcset="https://st.1001fonts.net/users/letterara/avatar" type="image/avif"/>
     <img alt="Avatar: Letterara Studio" class="rounded-circle avatar" referrerpolicy="no-referrer" src="https://st.1001fonts.net/users/letterara/avatar.jpeg" style="width:18px;height:18px"/>
    </picture>
    <a href="/users/letterara/" rel="author">
     Letterara Studio
    </a>
   </small>
  </span>
  <div class="font-toolbar-wrapper d-flex align-items-center">
   <div class="font-toolbar btn-group btn-group-sm typeface-info-buttons">
    <a aria-disabled="true" aria-label="This font is free for&l

In [6]:
# Check download link structure
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--headless')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

test_url = "https://www.1001fonts.com/handwritten-fonts.html?page=1"
driver.get(test_url)
time.sleep(3)

soup = BeautifulSoup(driver.page_source, 'html.parser')
font_list = soup.find('ul', {'class': 'font-list', 'id': 'typeface-browsing-list'})

if font_list:
    font_items = font_list.find_all('li', {'class': 'font-list-item font-preview'})
    print(f"Found {len(font_items)} font items\n")
    
    for i, item in enumerate(font_items[:3]):  # Check first 3 items
        print(f"\n=== Font Item {i+1} ===")
        
        # Look for download button
        download_link = item.find('a', {'role': 'button', 'class': 'btn btn-success'})
        print(f"Download button with class 'btn btn-success': {download_link is not None}")
        
        # Try alternative selectors
        all_buttons = item.find_all('a', {'role': 'button'})
        print(f"All buttons with role='button': {len(all_buttons)}")
        
        # Look for success class
        success_buttons = item.find_all('a', class_=lambda x: x and 'btn-success' in x if x else False)
        print(f"Buttons with 'btn-success' class: {len(success_buttons)}")
        
        if len(success_buttons) > 0:
            print(f"Success button href: {success_buttons[0].get('href')}")
            print(f"Success button text: {success_buttons[0].get_text(strip=True)}")
        
        # Check for license info
        license_yes = item.find('a', class_=lambda x: x and 'license-yes' in x if x else False)
        license_no = item.find('a', class_=lambda x: x and 'license-no' in x if x else False)
        
        if license_yes:
            print("License: commercial use")
        elif license_no:
            print("License: personal use")
        else:
            print("License: unknown")

driver.quit()


Found 12 font items


=== Font Item 1 ===
Download button with class 'btn btn-success': False
All buttons with role='button': 2
Buttons with 'btn-success' class: 1
Success button href: /download/rustic-roadway-personal-use.zip
Success button text: Download
License: personal use

=== Font Item 2 ===
Download button with class 'btn btn-success': False
All buttons with role='button': 2
Buttons with 'btn-success' class: 1
Success button href: /download/priestacy.zip
Success button text: Download
License: personal use

=== Font Item 3 ===
Download button with class 'btn btn-success': False
All buttons with role='button': 2
Buttons with 'btn-success' class: 1
Success button href: /download/rockybilly.zip
Success button text: Download
License: personal use
